# PostgreSQL Silver — Exploratory Data Analysis

## Source

Bronze PostgreSQL datasets:

- users
- subscription
- payments

## Objective

Understand the actual Bronze data before applying any Silver transformations.

## Exploration areas

1. Schema and data types
2. Record counts
3. Sample records
4. Null and missing values
5. Identifier quality
6. Duplicate analysis
7. Domain/value validation
8. Timestamp validation
9. User → Subscription → Payment relationships
10. Intentional dirty data confirmation

## Rule

No data is transformed or removed during this notebook.

All Silver decisions must be based on findings from the active Bronze data.

In [ ]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 01:30:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
POSTGRES_RUN_PATH = (
    "s3a://end-to-end-streaming-data-platform-bronze/"
    "postgres/ingestion_date=2026-09-01-Sep"
)

In [2]:
POSTGRES_RUN_PATH = (
    "s3a://end-to-end-streaming-data-platform-bronze/"
    "postgres/ingestion_date=2026-09-29-Sep"
)

In [3]:
df_users = spark.read.parquet(
    f"{POSTGRES_RUN_PATH}/users.parquet"
)

df_subscriptions = spark.read.parquet(
    f"{POSTGRES_RUN_PATH}/subscription.parquet"
)

df_payments = spark.read.parquet(
    f"{POSTGRES_RUN_PATH}/payments.parquet"
)

26/09/30 01:31:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Schema Inspection

In [4]:
print("=" * 70)
print("USERS SCHEMA")
print("=" * 70)

df_users.printSchema()


print("=" * 70)
print("SUBSCRIPTIONS SCHEMA")
print("=" * 70)

df_subscriptions.printSchema()


print("=" * 70)
print("PAYMENTS SCHEMA")
print("=" * 70)

df_payments.printSchema()

USERS SCHEMA
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_login: timestamp (nullable = true)

SUBSCRIPTIONS SCHEMA
root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)

PAYMENTS SCHEMA
root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)



In [4]:
print("=" * 70)
print("USERS SCHEMA")
print("=" * 70)

df_users.printSchema()


print("=" * 70)
print("SUBSCRIPTIONS SCHEMA")
print("=" * 70)

df_subscriptions.printSchema()


print("=" * 70)
print("PAYMENTS SCHEMA")
print("=" * 70)

df_payments.printSchema()

USERS SCHEMA
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_login: timestamp (nullable = true)

SUBSCRIPTIONS SCHEMA
root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)

PAYMENTS SCHEMA
root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)



Record Counts

In [5]:
print("=" * 70)
print("POSTGRES BRONZE — RECORD COUNTS")
print("=" * 70)

print(
    "Users:",
    df_users.count()
)

print(
    "Subscriptions:",
    df_subscriptions.count()
)

print(
    "Payments:",
    df_payments.count()
)

POSTGRES BRONZE — RECORD COUNTS


Users: 10000
Subscriptions: 20032
Payments: 11771


In [5]:
print("=" * 70)
print("POSTGRES BRONZE — RECORD COUNTS")
print("=" * 70)

print(
    "Users:",
    df_users.count()
)

print(
    "Subscriptions:",
    df_subscriptions.count()
)

print(
    "Payments:",
    df_payments.count()
)

POSTGRES BRONZE — RECORD COUNTS


Users: 10000
Subscriptions: 20141
Payments: 11767


Sample Records

In [6]:
print("=" * 70)
print("USERS SAMPLE")
print("=" * 70)

df_users.show(
    5,
    truncate=False
)


print("=" * 70)
print("SUBSCRIPTIONS SAMPLE")
print("=" * 70)

df_subscriptions.show(
    5,
    truncate=False
)


print("=" * 70)
print("PAYMENTS SAMPLE")
print("=" * 70)

df_payments.show(
    5,
    truncate=False
)
print("=" * 70)

USERS SAMPLE


+------------------------------------+-------------------------------+-------+--------------+-------------------+-------------------+
|user_id                             |email                          |country|account_status|created_at         |last_login         |
+------------------------------------+-------------------------------+-------+--------------+-------------------+-------------------+
|01a05eac-d160-76b8-a422-ab1e28713490|NULL                           |MAR    |active        |2026-01-01 00:00:00|NULL               |
|01a05eac-d162-7373-90a5-8795a88d4449|lucasstephanie@example.net     |XXX    |active        |2026-01-01 00:00:00|NULL               |
|01a05eac-d164-70cf-91eb-d988d5f5d901|zhatfield@example.net          |KSA    |active        |2026-01-01 00:00:00|2025-12-31 00:00:00|
|01a05eac-d166-7706-99d3-2f204ba5f333|  richardsonerica@example.org  |MAR    |active        |2026-01-01 00:00:00|NULL               |
|01a05eac-d168-7c30-9c39-27cda5a3167c|fsmith@example.org      

In [6]:
print("=" * 70)
print("USERS SAMPLE")
print("=" * 70)

df_users.show(
    5,
    truncate=False
)


print("=" * 70)
print("SUBSCRIPTIONS SAMPLE")
print("=" * 70)

df_subscriptions.show(
    5,
    truncate=False
)


print("=" * 70)
print("PAYMENTS SAMPLE")
print("=" * 70)

df_payments.show(
    5,
    truncate=False
)
print("=" * 70)

USERS SAMPLE


+------------------------------------+----------------------------+-------+--------------+-------------------+-------------------+
|user_id                             |email                       |country|account_status|created_at         |last_login         |
+------------------------------------+----------------------------+-------+--------------+-------------------+-------------------+
|01a0ee6e-4439-78c0-a27d-2792c5692db6|NULL                        |MAR    |active        |2026-01-01 00:00:00|NULL               |
|01a0ee6e-443b-7f1b-8c28-9b4fcd601db3|imorales@example.net        |XXX    |active        |2026-01-01 00:37:15|NULL               |
|01a0ee6e-443d-7035-af10-0f1594be2c6c|pjenkins@example.net        |EGY    |active        |2026-01-01 00:55:20|2025-12-31 00:55:20|
|01a0ee6e-443f-7384-b291-7a7a6476d53d|  robertsdavid@example.org  |KWT    |active        |2026-01-01 01:09:42|NULL               |
|01a0ee6e-4441-72a4-9427-3524994c6062|david86@example.net         |KWT    |active  

Null Analysis

In [7]:
def null_report(df):

    return df.select([
        F.sum(
            F.when(
                F.col(column).isNull(),
                1
            ).otherwise(0)
        ).alias(column)

        for column in df.columns
    ])

In [9]:
print("=" * 70)
print("USERS — NULL REPORT")
print("=" * 70)

null_report(
    df_users
).show(
    truncate=False
)


print("=" * 70)
print("SUBSCRIPTIONS — NULL REPORT")
print("=" * 70)

null_report(
    df_subscriptions
).show(
    truncate=False
)


print("=" * 70)
print("PAYMENTS — NULL REPORT")
print("=" * 70)

null_report(
    df_payments
).show(
    truncate=False
)
print("=" * 70)

USERS — NULL REPORT
+-------+-----+-------+--------------+----------+----------+
|user_id|email|country|account_status|created_at|last_login|
+-------+-----+-------+--------------+----------+----------+
|0      |1    |0      |0             |0         |9999      |
+-------+-----+-------+--------------+----------+----------+

SUBSCRIPTIONS — NULL REPORT
+---------------+-------+---------+----------+--------+---------+----------+
|subscription_id|user_id|plan_type|start_date|end_date|is_active|auto_renew|
+---------------+-------+---------+----------+--------+---------+----------+
|0              |0      |0        |0         |0       |0        |0         |
+---------------+-------+---------+----------+--------+---------+----------+

PAYMENTS — NULL REPORT
+----------+---------------+------+--------+------------+--------------+
|payment_id|subscription_id|amount|currency|payment_date|payment_status|
+----------+---------------+------+--------+------------+--------------+
|0         |0     

In [8]:
print("=" * 70)
print("USERS — NULL REPORT")
print("=" * 70)

null_report(
    df_users
).show(
    truncate=False
)


print("=" * 70)
print("SUBSCRIPTIONS — NULL REPORT")
print("=" * 70)

null_report(
    df_subscriptions
).show(
    truncate=False
)


print("=" * 70)
print("PAYMENTS — NULL REPORT")
print("=" * 70)

null_report(
    df_payments
).show(
    truncate=False
)
print("=" * 70)

USERS — NULL REPORT
+-------+-----+-------+--------------+----------+----------+
|user_id|email|country|account_status|created_at|last_login|
+-------+-----+-------+--------------+----------+----------+
|0      |1    |0      |0             |0         |9999      |
+-------+-----+-------+--------------+----------+----------+

SUBSCRIPTIONS — NULL REPORT
+---------------+-------+---------+----------+--------+---------+----------+
|subscription_id|user_id|plan_type|start_date|end_date|is_active|auto_renew|
+---------------+-------+---------+----------+--------+---------+----------+
|0              |0      |0        |0         |0       |0        |0         |
+---------------+-------+---------+----------+--------+---------+----------+

PAYMENTS — NULL REPORT
+----------+---------------+------+--------+------------+--------------+
|payment_id|subscription_id|amount|currency|payment_date|payment_status|
+----------+---------------+------+--------+------------+--------------+
|0         |0     

Identifier Quality

In [10]:
#1 Users
print("=" * 70)
print("USERS — ID QUALITY")
print("=" * 70)

df_users.select(
    F.count("*").alias(
        "total_rows"
    ),

    F.countDistinct(
        "user_id"
    ).alias(
        "distinct_user_ids"
    ),

    F.sum(
        F.when(
            F.col("user_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_user_ids"
    ),
).show()
print("=" * 70)

USERS — ID QUALITY


[Stage 24:>                                                         (0 + 1) / 1]

+----------+-----------------+-------------+
|total_rows|distinct_user_ids|null_user_ids|
+----------+-----------------+-------------+
|     10000|            10000|            0|
+----------+-----------------+-------------+



In [9]:
#1 Users
print("=" * 70)
print("USERS — ID QUALITY")
print("=" * 70)

df_users.select(
    F.count("*").alias(
        "total_rows"
    ),

    F.countDistinct(
        "user_id"
    ).alias(
        "distinct_user_ids"
    ),

    F.sum(
        F.when(
            F.col("user_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_user_ids"
    ),
).show()
print("=" * 70)

USERS — ID QUALITY


[Stage 24:>                                                         (0 + 1) / 1]

+----------+-----------------+-------------+
|total_rows|distinct_user_ids|null_user_ids|
+----------+-----------------+-------------+
|     10000|            10000|            0|
+----------+-----------------+-------------+



In [11]:
#1 Subscriptions
print("=" * 70)
print("SUBSCRIPTIONS — ID QUALITY")
print("=" * 70)

df_subscriptions.select(
    F.count("*").alias(
        "total_rows"
    ),

    F.countDistinct(
        "subscription_id"
    ).alias(
        "distinct_subscription_ids"
    ),

    F.sum(
        F.when(
            F.col("subscription_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_subscription_ids"
    ),

    F.sum(
        F.when(
            F.col("user_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_user_ids"
    ),
).show()
print("=" * 70)

SUBSCRIPTIONS — ID QUALITY
+----------+-------------------------+---------------------+-------------+
|total_rows|distinct_subscription_ids|null_subscription_ids|null_user_ids|
+----------+-------------------------+---------------------+-------------+
|     20032|                    20032|                    0|            0|
+----------+-------------------------+---------------------+-------------+



In [10]:
#1 Subscriptions
print("=" * 70)
print("SUBSCRIPTIONS — ID QUALITY")
print("=" * 70)

df_subscriptions.select(
    F.count("*").alias(
        "total_rows"
    ),

    F.countDistinct(
        "subscription_id"
    ).alias(
        "distinct_subscription_ids"
    ),

    F.sum(
        F.when(
            F.col("subscription_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_subscription_ids"
    ),

    F.sum(
        F.when(
            F.col("user_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_user_ids"
    ),
).show()
print("=" * 70)

SUBSCRIPTIONS — ID QUALITY
+----------+-------------------------+---------------------+-------------+
|total_rows|distinct_subscription_ids|null_subscription_ids|null_user_ids|
+----------+-------------------------+---------------------+-------------+
|     20141|                    20141|                    0|            0|
+----------+-------------------------+---------------------+-------------+



In [12]:
#3 Payments
print("=" * 70)
print("PAYMENTS — ID QUALITY")
print("=" * 70)

df_payments.select(
    F.count("*").alias(
        "total_rows"
    ),

    F.countDistinct(
        "payment_id"
    ).alias(
        "distinct_payment_ids"
    ),

    F.sum(
        F.when(
            F.col("payment_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_payment_ids"
    ),

    F.sum(
        F.when(
            F.col("subscription_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_subscription_ids"
    ),
).show()
print("=" * 70)

PAYMENTS — ID QUALITY
+----------+--------------------+----------------+---------------------+
|total_rows|distinct_payment_ids|null_payment_ids|null_subscription_ids|
+----------+--------------------+----------------+---------------------+
|     11771|               11771|               0|                    0|
+----------+--------------------+----------------+---------------------+



Duplicate Analysis

In [11]:
#3 Payments
print("=" * 70)
print("PAYMENTS — ID QUALITY")
print("=" * 70)

df_payments.select(
    F.count("*").alias(
        "total_rows"
    ),

    F.countDistinct(
        "payment_id"
    ).alias(
        "distinct_payment_ids"
    ),

    F.sum(
        F.when(
            F.col("payment_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_payment_ids"
    ),

    F.sum(
        F.when(
            F.col("subscription_id").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "null_subscription_ids"
    ),
).show()
print("=" * 70)

PAYMENTS — ID QUALITY
+----------+--------------------+----------------+---------------------+
|total_rows|distinct_payment_ids|null_payment_ids|null_subscription_ids|
+----------+--------------------+----------------+---------------------+
|     11767|               11767|               0|                    0|
+----------+--------------------+----------------+---------------------+



In [25]:
print("=" * 70)
print("DUPLICATE PRIMARY IDS")
print("=" * 70)


print(
    "Duplicate user_ids:",
    df_users.groupBy(
        "user_id"
    ).count().filter(
        F.col("count") > 1
    ).count()
)


print(
    "Duplicate subscription_ids:",
    df_subscriptions.groupBy(
        "subscription_id"
    ).count().filter(
        F.col("count") > 1
    ).count()
)


print(
    "Duplicate payment_ids:",
    df_payments.groupBy(
        "payment_id"
    ).count().filter(
        F.col("count") > 1
    ).count()
)
print("=" * 70)

DUPLICATE PRIMARY IDS
Duplicate user_ids: 0
Duplicate subscription_ids: 0
Duplicate payment_ids: 0


Domain Exploration

In [12]:
print("=" * 70)
print("DUPLICATE PRIMARY IDS")
print("=" * 70)


print(
    "Duplicate user_ids:",
    df_users.groupBy(
        "user_id"
    ).count().filter(
        F.col("count") > 1
    ).count()
)


print(
    "Duplicate subscription_ids:",
    df_subscriptions.groupBy(
        "subscription_id"
    ).count().filter(
        F.col("count") > 1
    ).count()
)


print(
    "Duplicate payment_ids:",
    df_payments.groupBy(
        "payment_id"
    ).count().filter(
        F.col("count") > 1
    ).count()
)
print("=" * 70)

DUPLICATE PRIMARY IDS
Duplicate user_ids: 0
Duplicate subscription_ids: 0
Duplicate payment_ids: 0


In [13]:
#1 Users
print("=" * 70)
print("USERS — COUNTRY DISTRIBUTION")
print("=" * 70)

df_users.groupBy(
    "country"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)


print("=" * 70)
print("USERS — ACCOUNT STATUS")
print("=" * 70)

df_users.groupBy(
    "account_status"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)
print("=" * 70)

USERS — COUNTRY DISTRIBUTION
+-------+-----+
|country|count|
+-------+-----+
|MAR    |1510 |
|QAT    |1452 |
|KWT    |1431 |
|JOR    |1421 |
|EGY    |1400 |
|UAE    |1394 |
|KSA    |1391 |
|XXX    |1    |
+-------+-----+

USERS — ACCOUNT STATUS
+--------------+-----+
|account_status|count|
+--------------+-----+
|active        |10000|
+--------------+-----+



Subscriptions

In [13]:
#1 Users
print("=" * 70)
print("USERS — COUNTRY DISTRIBUTION")
print("=" * 70)

df_users.groupBy(
    "country"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)


print("=" * 70)
print("USERS — ACCOUNT STATUS")
print("=" * 70)

df_users.groupBy(
    "account_status"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)
print("=" * 70)

USERS — COUNTRY DISTRIBUTION
+-------+-----+
|country|count|
+-------+-----+
|JOR    |1468 |
|MAR    |1465 |
|UAE    |1439 |
|EGY    |1437 |
|QAT    |1413 |
|KSA    |1402 |
|KWT    |1375 |
|XXX    |1    |
+-------+-----+

USERS — ACCOUNT STATUS
+--------------+-----+
|account_status|count|
+--------------+-----+
|active        |10000|
+--------------+-----+



In [14]:
print("=" * 70)
print("SUBSCRIPTIONS — PLAN TYPE")
print("=" * 70)

df_subscriptions.groupBy(
    "plan_type"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)


print("=" * 70)
print("SUBSCRIPTIONS — ACTIVE STATUS")
print("=" * 70)

df_subscriptions.groupBy(
    "is_active"
).count().show()


print("=" * 70)
print("SUBSCRIPTIONS — AUTO RENEW")
print("=" * 70)

df_subscriptions.groupBy(
    "auto_renew"
).count().show()
print("=" * 70)

SUBSCRIPTIONS — PLAN TYPE
+---------+-----+
|plan_type|count|
+---------+-----+
|free     |10017|
|premium  |10015|
+---------+-----+

SUBSCRIPTIONS — ACTIVE STATUS
+---------+-----+
|is_active|count|
+---------+-----+
|     true|20032|
+---------+-----+

SUBSCRIPTIONS — AUTO RENEW
+----------+-----+
|auto_renew|count|
+----------+-----+
|     false|14045|
|      true| 5987|
+----------+-----+



In [14]:
print("=" * 70)
print("SUBSCRIPTIONS — PLAN TYPE")
print("=" * 70)

df_subscriptions.groupBy(
    "plan_type"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)


print("=" * 70)
print("SUBSCRIPTIONS — ACTIVE STATUS")
print("=" * 70)

df_subscriptions.groupBy(
    "is_active"
).count().show()


print("=" * 70)
print("SUBSCRIPTIONS — AUTO RENEW")
print("=" * 70)

df_subscriptions.groupBy(
    "auto_renew"
).count().show()
print("=" * 70)

SUBSCRIPTIONS — PLAN TYPE
+---------+-----+
|plan_type|count|
+---------+-----+
|premium  |10080|
|free     |10061|
+---------+-----+

SUBSCRIPTIONS — ACTIVE STATUS
+---------+-----+
|is_active|count|
+---------+-----+
|     true|20141|
+---------+-----+

SUBSCRIPTIONS — AUTO RENEW
+----------+-----+
|auto_renew|count|
+----------+-----+
|     false|14115|
|      true| 6026|
+----------+-----+



In [15]:
# 3Payments
print("=" * 70)
print("PAYMENTS — STATUS")
print("=" * 70)

df_payments.groupBy(
    "payment_status"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)


print("=" * 70)
print("PAYMENTS — CURRENCY")
print("=" * 70)

df_payments.groupBy(
    "currency"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)
print("=" * 70)

PAYMENTS — STATUS
+--------------+-----+
|payment_status|count|
+--------------+-----+
|success       |9987 |
|failed        |1784 |
+--------------+-----+

PAYMENTS — CURRENCY
+--------+-----+
|currency|count|
+--------+-----+
|USD     |11771|
+--------+-----+



Timestamp Validation

In [15]:
# 3Payments
print("=" * 70)
print("PAYMENTS — STATUS")
print("=" * 70)

df_payments.groupBy(
    "payment_status"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)


print("=" * 70)
print("PAYMENTS — CURRENCY")
print("=" * 70)

df_payments.groupBy(
    "currency"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)
print("=" * 70)

PAYMENTS — STATUS
+--------------+-----+
|payment_status|count|
+--------------+-----+
|success       |10052|
|failed        |1715 |
+--------------+-----+

PAYMENTS — CURRENCY
+--------+-----+
|currency|count|
+--------+-----+
|USD     |11767|
+--------+-----+



In [16]:
#1 Users
print("=" * 70)
print("USERS — TIMESTAMP VALIDATION")
print("=" * 70)

login_before_creation = df_users.filter(
    F.col("last_login").isNotNull()
    &
    (
        F.col("last_login")
        <
        F.col("created_at")
    )
).count()

print(
    "last_login before created_at:",
    login_before_creation
)
print("=" * 70)

USERS — TIMESTAMP VALIDATION
last_login before created_at: 1


In [16]:
#1 Users
print("=" * 70)
print("USERS — TIMESTAMP VALIDATION")
print("=" * 70)

login_before_creation = df_users.filter(
    F.col("last_login").isNotNull()
    &
    (
        F.col("last_login")
        <
        F.col("created_at")
    )
).count()

print(
    "last_login before created_at:",
    login_before_creation
)
print("=" * 70)

USERS — TIMESTAMP VALIDATION
last_login before created_at: 1


In [17]:
#2 Subscriptions
print("=" * 70)
print("SUBSCRIPTIONS — TIMESTAMP VALIDATION")
print("=" * 70)

end_before_start = df_subscriptions.filter(
    F.col("end_date").isNotNull()
    &
    (
        F.col("end_date")
        <
        F.col("start_date")
    )
).count()

print(
    "end_date before start_date:",
    end_before_start
)
print("=" * 70)

SUBSCRIPTIONS — TIMESTAMP VALIDATION
end_date before start_date: 1


Relationships

In [17]:
#2 Subscriptions
print("=" * 70)
print("SUBSCRIPTIONS — TIMESTAMP VALIDATION")
print("=" * 70)

end_before_start = df_subscriptions.filter(
    F.col("end_date").isNotNull()
    &
    (
        F.col("end_date")
        <
        F.col("start_date")
    )
).count()

print(
    "end_date before start_date:",
    end_before_start
)
print("=" * 70)

SUBSCRIPTIONS — TIMESTAMP VALIDATION
end_date before start_date: 1


In [18]:
# User → Subscription

print("=" * 70)
print("USER → SUBSCRIPTION RELATIONSHIP")
print("=" * 70)

orphan_subscriptions = (
    df_subscriptions
    .join(
        df_users.select(
            "user_id"
        ),
        on="user_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan subscriptions:",
    orphan_subscriptions
)
print("=" * 70)

USER → SUBSCRIPTION RELATIONSHIP
Orphan subscriptions: 0


In [18]:
# User → Subscription

print("=" * 70)
print("USER → SUBSCRIPTION RELATIONSHIP")
print("=" * 70)

orphan_subscriptions = (
    df_subscriptions
    .join(
        df_users.select(
            "user_id"
        ),
        on="user_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan subscriptions:",
    orphan_subscriptions
)
print("=" * 70)

USER → SUBSCRIPTION RELATIONSHIP
Orphan subscriptions: 0


In [19]:
# Subscription → Payment

print("=" * 70)
print("SUBSCRIPTION → PAYMENT RELATIONSHIP")
print("=" * 70)

orphan_payments = (
    df_payments
    .join(
        df_subscriptions.select(
            "subscription_id"
        ),
        on="subscription_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan payments:",
    orphan_payments
)
print("=" * 70)

SUBSCRIPTION → PAYMENT RELATIONSHIP
Orphan payments: 0


Cardinality Exploration

In [19]:
# Subscription → Payment

print("=" * 70)
print("SUBSCRIPTION → PAYMENT RELATIONSHIP")
print("=" * 70)

orphan_payments = (
    df_payments
    .join(
        df_subscriptions.select(
            "subscription_id"
        ),
        on="subscription_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan payments:",
    orphan_payments
)
print("=" * 70)

SUBSCRIPTION → PAYMENT RELATIONSHIP
Orphan payments: 0


In [21]:
print("=" * 70)
print("USER → SUBSCRIPTION CARDINALITY")
print("=" * 70)

df_subscriptions.groupBy(
    "user_id"
).count().groupBy(
    "count"
).count().orderBy(
    "count"
).show()
print("=" * 70)

USER → SUBSCRIPTION CARDINALITY
+-----+-----+
|count|count|
+-----+-----+
|    1| 3311|
|    2| 3346|
|    3| 3343|
+-----+-----+



In [20]:
print("=" * 70)
print("USER → SUBSCRIPTION CARDINALITY")
print("=" * 70)

df_subscriptions.groupBy(
    "user_id"
).count().groupBy(
    "count"
).count().orderBy(
    "count"
).show()
print("=" * 70)

USER → SUBSCRIPTION CARDINALITY
+-----+-----+
|count|count|
+-----+-----+
|    1| 3288|
|    2| 3283|
|    3| 3429|
+-----+-----+



In [22]:
print("=" * 70)
print("SUBSCRIPTION → PAYMENT CARDINALITY")
print("=" * 70)

df_payments.groupBy(
    "subscription_id"
).count().groupBy(
    "count"
).count().orderBy(
    "count"
).show()
print("=" * 70)

SUBSCRIPTION → PAYMENT CARDINALITY
+-----+-----+
|count|count|
+-----+-----+
|    1| 8485|
|    2| 1304|
|    3|  226|
+-----+-----+



Payment Amount Exploration

In [21]:
print("=" * 70)
print("SUBSCRIPTION → PAYMENT CARDINALITY")
print("=" * 70)

df_payments.groupBy(
    "subscription_id"
).count().groupBy(
    "count"
).count().orderBy(
    "count"
).show()
print("=" * 70)

SUBSCRIPTION → PAYMENT CARDINALITY
+-----+-----+
|count|count|
+-----+-----+
|    1| 8637|
|    2| 1199|
|    3|  244|
+-----+-----+



In [23]:
print("=" * 70)
print("PAYMENT AMOUNT SUMMARY")
print("=" * 70)

df_payments.select(
    F.min("amount").alias(
        "min_amount"
    ),

    F.max("amount").alias(
        "max_amount"
    ),

    F.avg("amount").alias(
        "avg_amount"
    ),
).show()
print("=" * 70)

PAYMENT AMOUNT SUMMARY
+--------------------+--------------------+--------------------+
|          min_amount|          max_amount|          avg_amount|
+--------------------+--------------------+--------------------+
|9.990000000000000000|29.99000000000000...|18.23313992014272...|
+--------------------+--------------------+--------------------+



In [22]:
print("=" * 70)
print("PAYMENT AMOUNT SUMMARY")
print("=" * 70)

df_payments.select(
    F.min("amount").alias(
        "min_amount"
    ),

    F.max("amount").alias(
        "max_amount"
    ),

    F.avg("amount").alias(
        "avg_amount"
    ),
).show()
print("=" * 70)

PAYMENT AMOUNT SUMMARY
+--------------------+--------------------+--------------------+
|          min_amount|          max_amount|          avg_amount|
+--------------------+--------------------+--------------------+
|9.990000000000000000|29.99000000000000...|18.16413104444633...|
+--------------------+--------------------+--------------------+



# EDA Summary

## Data explored

- users
- subscription
- payments

## Verified

- Schema and active data types
- Record counts
- Null distribution
- Identifier uniqueness
- Duplicate primary IDs
- Domain distributions
- Timestamp consistency
- User → Subscription relationships
- Subscription → Payment relationships
- Relationship cardinality
- Payment amount distribution

## dirty data confirmed

- Missing user email
- Invalid user country
- last_login before created_at
- Whitespace around one email
- Subscription end_date before start_date

## Transformation decisions

Not made in this notebook.

All Silver transformation decisions will be documented
before implementation in the Transformation notebook.